# 06 — Generalization Gap Investigation

Recorded external result: Kaggle public leaderboard **75.358%** vs internal 5-fold CV **84.40%** (gap **9.042 pp** at display precision). The score was recorded once, after modeling, and was never used for any decision — no test labels were inferred, no leaderboard tuning was performed.

This notebook recomputes nothing expensive: it loads the measured investigation artefacts (`reports/gap_*.json`), asserts their mutual consistency, and documents the evidence-backed conclusion.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
E = json.loads((ROOT/'reports'/'extended_results.json').read_text())
W1 = json.loads((ROOT/'reports'/'gap_reproduction.json').read_text())
W2 = json.loads((ROOT/'reports'/'gap_dist_shift.json').read_text())
W3 = json.loads((ROOT/'reports'/'gap_groupcv.json').read_text())
W4 = json.loads((ROOT/'reports'/'gap_robustness.json').read_text())
W5 = json.loads((ROOT/'reports'/'gap_stability.json').read_text())
print('artefacts loaded')

## Consistency gates (fail loudly on any drift)


In [ ]:
assert W1['matches_preserved_bool'] and abs(W1['mean'] - R['best_cv']) < 1e-9
assert W1['submission']['exact_match_bool']
g3 = next(r for r in W3['results'] if r['strategy'] == 'group_ticket')
assert abs(g3['mean'] - W4['feature_matrix'][-1]['group_mean']) < 1e-12, 'W3 vs W4 ticket mismatch'
gap = round(100*R['best_cv'], 2) - 100*E['kaggle']['score']
assert abs(E['kaggle']['score'] - 0.75358) < 1e-9 and abs(E['kaggle']['gap_vs_best_cv_pp'] - 9.042) < 1e-9
print(f"gates pass: reproduction exact, W3==W4 ticket={g3['mean']:.4f}, gap={E['kaggle']['gap_vs_best_cv_pp']:.3f} pp")

## Random CV vs group-aware CV
Ticket-group CV (0.7991) sits ~4.5 pp below random CV; surname (0.8158) and family-ticket (0.8137) between. Overlap is consistent with the mechanism and provides a plausible pathway: ~34% of validation tickets and ~36% of surnames also appear in training folds.


In [ ]:
print(f"{'strategy':<22}{'mean':>8}{'std':>8}")
print(f"{'random (original)':<22}{R['best_cv']:>8.4f}{R.get('ablation_full_tuned',{}).get('std',0.0141):>8.4f}")
for r in W3['results']:
    import numpy as np
    print(f"{r['strategy']:<22}{r['mean']:>8.4f}{float(np.std(r['folds'])):>8.4f}")
print('overlap: ticket %.1f%%, surname %.1f%%' % (W3['overlap']['ticket_mean'], W3['overlap']['surname_mean']))

## Shift, seeds, robustness (headlines)
Core-variable shift is mild (Sex TV 0.011, Fare KS 0.039); the largest shifts are TicketGroupSize/FarePerPerson — expected artefacts of leakage-free fitting. Seeds span 83.06–84.40 (seed-42 highest, range 1.3 pp); repeated 5×5 CV gives 83.52 ± 2.51. Model ranking flips under ticket-group CV (LR 0.8238 > RF 0.7991).


In [ ]:
print('KS top:', [(d['feature'], round(d['value'],3)) for d in W2['ks_ranked'][:3]])
print('TV top:', [(d['feature'], round(d['value'],3)) for d in W2['tv_ranked'][:3]])
print('seeds:', [(s['seed'], round(s['mean'],4)) for s in W5['seeds']], 'repeated:', {k: round(v,4) for k,v in W5['repeated'].items()})
print('models random->group:', [(m['model'], round(m['random_mean'],4), round(m['group_mean'],4)) for m in W4['models']])
print('Brier:', round(W5['brier'],4), '| confidence train/test:', round(W5['confidence']['train_mean_max_proba'],4), round(W5['confidence']['test_mean_max_proba'],4))

## Diagnostic conclusion

**Case E — no single explanation is sufficient**, with Case A (group-overlap optimism) the largest measured contributor:

- Group-aware CV (79.9–81.6%) is much closer to Kaggle (75.36%) than random CV (84.40%), and the overlap mechanism is measured (~34–36% cross-fold group overlap of tickets/surnames (dependence, not target leakage; cf. leakage audit L1-L7 PASS)). But ~4.5 pp of gap remains unexplained by grouping alone.
- Train/test shift on core variables is mild and confidence levels match (0.8475 vs 0.8500) — shift alone (Case B) cannot carry the gap.
- Ticket-derived features degrade most under group CV (partial Case C), E→F group gain is only +0.0022 (0.79911→0.80133); best under ticket-group CV is simplest set A (0.82377), not F — the cabin win does not transfer to grouped evaluation.
- Seed range (1.3 pp) and repeated CV (83.52) show seed-42 slightly flattering but not outlying — instability (Case D) is minor.

## Model decision: Outcome A + robustness reporting (Outcome C, complementary)

**Retain the tuned Random Forest** under the pre-registered primary methodology (random stratified CV, 84.40%). Rationale: model selection must use internal methodology, not the external score; even the best group-CV estimate (LR 82.38%) overshoots Kaggle by ~7 pp, so group CV is not a calibrated Kaggle predictor and switching models on its basis would be unjustified. RF is methodology-locked, not robustness-optimal: its random→group drop (0.0449) is ~10× LR's (0.0045), and the ranking reversal is acknowledged — and switching to *match* 75.36% would be leaderboard reasoning. Group-aware ticket CV (79.91%) is added as a reported secondary robustness context, not a replacement.
